# Common designs flaws
- Deadlock
- Livelock
- Underspecification
- Overspecification
- Violations of constraints
- Assumptions about speed

Welcome, everyone. When we design and build complex concurrent systems, we inevitably run into serious, hidden design flaws. As you can see on the screen, we're talking about critical issues like deadlocks, livelocks, underspecification, and incorrect assumptions about speed.

Catching these errors through standard testing is incredibly difficult because they often depend on highly specific, unpredictable timings. So, how can we mathematically guarantee that our system behaves exactly as intended and avoids these violations?

The answer is Model Checking. Today, we are going to explore how we can use this automated verification approach to catch these fatal flaws before they ever make it into production.

# What is Model Checking
Model checking tools automatically verify whether:
$$M \models \varphi$$
holds, where M is a (finite-state) model of a system and property φ is stated in some formal notation.



Model checking is...

But there's a catch. When we try to comprehensively verify a system, we quickly run into the 'State-Explosion' problem.

## Problem
State-Explosion problem

Systems get too large to check every single possibility.

# Verification vs. Debugging
- **Verification** approach: tries to ascertain the correctness of a detailed model $M$ of the system under validation   
- **Debuggings** approach: tries to find errors in a model $M$

Model checking is most effective in combination with the *debugging* approach.



That is why model checking is actually most effective when combined with a debugging approach. Instead of trying to prove absolute perfection from the start, we use it as a highly targeted tool to hunt down and find errors in our detailed models.

# SPIN
Is one of the most powerful model checkers

To perform this kind of advanced debugging, we use one of the most powerful model checkers available today: SPIN

**S**imple **P**romela **IN**terpreter

Is a tool for analysing the logical conisistency of concurrent systems, specifically of data communication protocols.

Concurrent systems are described in the modelling language called **Promela**.

To communicate with SPIN, we write our system descriptions in a language called PROMELA

## PROMELA (= Protocol/Process Meta Language)
- allows for the dynamic creation of concurrent processes.
-  communication via message channels can be defined to be
    - synchronous (i.e. rendezvous), or
    - asynchronous (i.e. buffered).
- resembles the programming language C
- specification language to model finite-state systems


If you are familiar with the C programming language, PROMELA will look very similar. It gives us the power to dynamically create concurrent processes and define exactly how they communicate.

# Promela Model

### Model Components

A Promela model mainly consists of the following parts:

**1. Type, Channel, and Variable declarations**
```c
mtype = {MSG, ACK};     /* TYPE     declaration */
chan toS = ...          /* CHANNEL  declaration */
chan toR = ...          /* CHANNEL  declaration */
bool flag;              /* VARIABLE declaration */

proctype Sender() {     /* PROCESS  declaration */
    ...                 /* process  body */
}

proctype Receiver() {
    ...
}

init {                  /* init     process*/ 
    ...                 /* creates  processes */
}
```


there is one strict rule you must remember: A Promela model maps to a finite transition system.



A **Promela model** corresponds with a (usually very large, but) **finite transition system**, so it strictly requires:
* no unbounded **data**
* no unbounded **channels**
* no unbounded **processes**
* no unbounded **process creation**

This means absolutely everything must be strictly bounded

# Processes
A process type (```proctype```) consists of:
- name
- formal parameters
- local variable declarations 
- body (sequence of statements)

```c
proctype Sender(chan in; chan out) {
    bit sndB, rcvB;
    do
    :: out ! MSG, sndB ->
        in ? ACK, rcvB;
        if
        :: sndB == rcvB -> sndB = 1-sndB
        :: else -> skip
        fi
    od
}
```




**Overview** of a process:
- ```proctype```
- executes **concurrently** with all other processes
- **communicate** with other processes using:
    - global (shared) variables
    - channels
- several processes of the same type with their own local state



# Process automaton
Every promela `proctype` defines a **finite state automaton**, (S, s0, L, T, F), where:
- S is a set of states
- s0 is the initial state, s0 ∈ S
- L is a finite set of labels
- T is a set of transitions, T ⊆ S × L × S
- F is a set of final states, F ⊆ S

## Creating and starting processes:
```c
proctype Foo(byte x) {
    ...
}

init {
    int pid2 = run Foo(2);
    run Foo(27);            /* creates and returns pid */
}

active[3] proctype Bar() {  /* creates 3 processes (optional) */
    ...
}
```
- can be executed at any point in the execution (within any process)
- start executing after the ```run``` statement

## Variables and types
### Basic types 
``` c
bit turn=1;     /* [0..1] */
bool flag;      /* [0..1] */
byte counter;   /* [0..255] */
short s;        /* [-216-1.. 216 –1] */
int msg;        /* [-232-1.. 232 –1] */

```

### Arrays
```c
byte a[27];
bit flags[4];
```

### Records
```c
typedef Record {
    short f1;
    byte f2;
}

Record rr;      /* variable declaration */
rr.f1 = ...
rr.f2 = ...
```

<div style="display: flex;">

<div style="flex: 50%; padding-right: 15px;">

## Statements
The body of a process consists of a **sequence of statements**.

A statement can be:
- **executable:** can be executed immediately
- **blocked:** cannot be executed

### Statements always executable:
- **Assignments** 
- ```skip```
- ```printf```
- ```assert(<expr>)```
    - if ```<expr>``` evaluates to zero -> **error**

### Statements executable if:
- **Expression:** if it evaluates to non-zero
- `run`: if a new process can be created (the number of processes is *bounded*)



</div>

<div style="flex: 50%; padding-left: 15px;">

<br><br>
```c
proctype Example() {
    byte x = 1;     /* Assignment (always executable) */
    
    skip;           /* skip (always executable) */
    
                    /* Expression (executable se non-zero) */
    (x > 0) -> 
        printf("x is positive\n");  /* printf (always executable) */
    assert(x == 1); /* assert (always executable) */
}

# Interleaving semantics
- Promela processes executes **concurrently**
- **Non-deterministic** scheduling of the processing
- Processes are **interleaved**
- All statements are **atomic**
- **non-deterministically** choices

# Mutual Exclusion: wrong

In [1]:
%%writefile ../examples/mutual_exclusion_wrong.pml
bit flag;       /* signal entering/leaving the section */
byte mutex;     /* procs in the critical section. */

proctype P(bit i) {
    flag != 1;  /* wait until the other process leaves the section */
    flag = 1;   /* PROBLEM: both processes can pass the condition at the same time*/
    mutex++;
    printf("MSC: P(%d) has entered section.\n", i);
    mutex--;
    flag = 0;
}

proctype monitor() {
    assert(mutex != 2);
}

init {
    atomic { run P(0); run P(1); run monitor(); }
}


Overwriting ../examples/mutual_exclusion_wrong.pml


Let's look at a classic problem: Mutual Exclusion. In our first example, two processes try to enter a critical section at the same time. They check a flag, set it, and increment a mutex variable. It looks fine at first glance. But when we run SPIN, it immediately finds an assertion violation! 

In [ ]:
%%bash
./../scripts/run_mutual_exclusion_wrong.sh


Why? Because both processes managed to read the flag at the exact same time before either could change it. To fix this, as you can see in the 'Right' example, we have to wrap the check and the assignment inside an atomic block. This tells SPIN that these steps must happen together, uninterrupted.

# Mutual Exclusion: right

In [ ]:
%%writefile ../examples/mutual_exclusion_right.pml
bit flag;       /* signal entering/leaving the section */
byte mutex;     /* procs in the critical section. */

proctype P(bit i) {
    atomic { flag != 1; flag = 1;};
    mutex++;
    printf("MSC: P(%d) has entered section.\n", i);
    mutex--;
    flag = 0;
}

proctype monitor() {
    assert(mutex != 2);
}

init {
    atomic { run P(0); run P(1); run monitor(); }
}


In [ ]:
%%bash
./../scripts/run_mutual_exclusion_right.sh


# Mutual Exclusion: can you spot the error?

In [ ]:
%%writefile ../examples/mutual_exclusion_wrong_2.pml
bit x, y;
byte mutex;

active proctype A(){
    x = 1;
    y == 0;
    mutex++;
    mutex--;
    x = 0; 
}

active proctype B(){
    y = 1;
    x == 0;
    mutex++;
    mutex--;
    y = 0; 
}

active proctype monitor(){
    assert(mutex != 2); 
}

init{
    atomic{run A(); run B(); run monitor();}
}


In [ ]:
%%bash
./../scripts/run_mutual_exclusion_wrong_2.sh


## if-statement

```c
if
:: choice1 -> stat1.1; stat1.2; stat1.3; ...
:: choice2 -> stat2.1; stat2.2; stat2.3; ...
:: ...
:: choicen -> statn.1; statn.2; statn.3; ...
:: else    -> ...
fi;
```

### non-deterministic branching

Give `n` a random value
```c
if
:: skip -> n=0
:: skip -> n=1
:: skip -> n=2
:: skip -> n=3
fi
```

## do-statement
```c
mtype = { RED, YELLOW, GREEN };

active proctype TrafficLight() {
    byte state = GREEN;
    do
    :: (state == GREEN)  -> state = YELLOW;
    :: (state == YELLOW) -> state = RED;
    :: (state == RED)    -> state = GREEN;
    od;
}
```
- with respect to the choices behaves in the same way as an if-statement
- a **do-statement** repeats the choice selection
- `break`


# Communication

<div align="center">
    <img src="images/communication.png" width="500">
</div>

Communication between processes is via **channels**:
- message passing
- randez-vous synchronization

Both defined as **channels**:
`chan <name> = [<dim>] of {<t1>, <t2>, ..., <tn>};`

dim == 0 is a special case **randez-vous**
t1,t2,...,tn type of elements that will be transmitted over the channel

# Communication
<div style="display: flex;">

<div style="flex: 60%; padding-right: 15px;">

### ! Sending (putting message into a channel)
`ch ! <expr1>, <expr2>, ... <exprn>;`
- executable if the channel is not full

### ? Receiving (getting a message out of a channel)
**Message Passing:**

`ch ? <var1>, <var2>, ... <varn>;`
- if the channel is **not empty**, the message is fetched from the channel and the individual parts of the message are stored into the `<vari>`s

**Message Testing:**

`ch ? <const1>, <const2>, ... <constn>;`
- If the channel is not empty and the message at the front of the channel evaluates to the individual `<consti>`, the statement is executable and the message is removed from the channel.


</div>


<div style="flex: 40%; padding-left: 15px;">

<br><br>
<img src="images/communication.png" width="100%">

# Communication
<div style="display: flex;">

<div style="flex: 60%; padding-right: 15px;">

### Randez-vous communication `<dim>==0`
The number of elements in the chanel is now **zero**
- If **send** `ch!` is enabled and if there is a corresponding **receive** `ch?` that can be executed simultaneously and the constants match, then both statements are enabled.
- Both statements will “handshake” and together take the transition.

**Example:**
- `chan ch = [0] of {bit, byte};`
- P wants to do `ch ! 1, 3+7`
- Q wants to do `ch ? 1, x`
- Then after the communication, `x` will have the value `10`.


</div>


<div style="flex: 40%; padding-left: 15px;">

<br><br>
<img src="images/communication.png" width="100%">

# Alternating Bit Protocol
- To every message, the sender adds a bit
- The receiver acknowledges each message by sending the received bit back
- To receiver only excepts messages with a bit that it excepted to receive
- If the sender is sure that the receiver has correctly received the previous message, it sends a new message and it alternates the accompanying bit.


<div style="display: flex;">

<div style="flex: 50%; padding-right: 15px;">

```c
mtype {MSG, ACK};
chan toS = [2] of {mtype, bit};
chan toR = [2] of {mtype, bit};

proctype Sender(chan in, out){
    bit sendbit, recvbit;
    do
    :: out ! MSG, sendbit ->
        in ? ACK, recvbit;
        if
        :: recvbit == sendbit ->
            sendbit = 1-sendbit
        :: else
        fi
    od
}
```



</div>


<div style="flex: 50%; padding-left: 15px;">

<br><br>
```c
proctype Receiver(chan in, out){
    bit recvbit;
    do
    :: in ? MSG, recvbit ->
        out ! ACK, recvbit;
    od
    }

init{
    run Sender(toS, toR);
    run Receiver(toR, toS);
}
```


## Statements

### Statements always executable:
- **Assignments** 
- ```skip```
- ```printf```
- ```assert(<expr>)```
    - if ```<expr>``` evaluates to zero -> **error**
- `break` (in a do-statement)

### Statements executable if:
- **Expression:** if it evaluates to non-zero
- `run`: if a new process can be created (the number of processes is *bounded*)
- `if`: if at least one guard is executable
- `do`: if at least one guard is executable
- `ch!`: if channel ch is not full
- `ch?`: if channel ch is not empty

## Atomic
`atomic {stat_1; stat_2; ...; stat_n}`
- can be used to group statements into an atomic sequence
- executable if `stat_1` is executable
- if `stat_i` is blocked, the **atomicity token** is (temporarily) lost and the other processes may do a step

To wrap up, I want to return to that 'State-Explosion' problem we mentioned earlier. Every tiny step your processes take generates new states for SPIN to check. To keep our models efficient, we use keywords like atomic and d_step. While atomic groups statements so they execute without interruption, d_step goes further. 

## d_step

<div style="display: flex;">

<div style="flex: 50%; padding-right: 15px;">

`d_step {stat_1; stat_2; ...; stat_n}`
- more efficient version of `atomic`: no intermediate states are generated and stored
- may only contain **deterministic** steps
- **run-time** error if `stat_i` blocks
- useful to perform intermediate computations in a single transition
- `atomic` and `d_step` can be used to lower the number of states of the model



</div>
<div style="flex: 50%; padding-left: 15px;">

```c
:: Rout?i(v) -> d_step {
            k++;
            e[k].ind = i;
            e[k].val = v;
            i=0; v=0 ;
        }
```


A d_step treats the entire sequence as a single, deterministic transition, generating no intermediate states at all. Using these tools wisely is the key to verifying complex systems without running out of memory.

<div style="display: flex;">

<div style="flex: 50%; padding-right: 15px;">

## NO Atomicity

```c
proctype P1() { t1a; t1b; t1c }
proctype P2() { t2a; t2b; t2c }
init { run P1(); run P2() }

```

<img src="images/Screenshot 2026-04-26 alle 15.08.33.png" width="100%">

</div>
<div style="flex: 50%; padding-left: 15px;">

## atomic

```c
proctype P1() { atomic {t1a; t1b; t1c} }
proctype P2() { t2a; t2b; t2c }
init { run P1(); run P2() }

```

<img src="images/Screenshot 2026-04-26 alle 15.09.46.png" width="100%">

</div>
<div style="flex: 50%; padding-left: 15px;">

## d_step

```c
proctype P1() { d_step {t1a; t1b; t1c} }
proctype P2() { t2a; t2b; t2c }
init { run P1(); run P2() }

```

<img src="images/Screenshot 2026-04-26 alle 15.10.10.png" width="100%">

# Checking for pure atomicity

Suppose we want to check that none of the atomic clauses in our model are ever blocked 


### 1. Add a global bit variable
```c
bit aflag;
```

### 2. Change all `atomic` clauses to:
```c
atomic {
    stat_1;
    aflag = 1;
    stat_2;
    stat_3
    ...
    stat_n;
    aflag = 0;
}
```

### 3. Check that `aflag` is always 0

```c
active proctype monitor {
    assert(!aflag);
}
```